# 12.8 頻譜怎麼變成特徵序列？


log-mel每一列是頻率區段，每一列橫向又有多個時間框；文字接口則希望一串位置，每個位置有固定特徵寬度。於是把頻譜轉一下軸，讓一個時間框成為一條向量，再用線性變換得到較短的特徵。這裡的序列位置來自時間，不是原始每個振幅樣本。

前置是[12.7的log-mel](https://birdhackor.github.io/tiny-perceptron-vlm/12.7.html)、[10.3的共享線性變換](https://birdhackor.github.io/tiny-perceptron-vlm/10.3.html)與[W.3軸順序](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.3)。輸入一批B段錄音，頻譜是[B,M,T]，M為mel區段數、T為框數；`transpose`後為[B,T,M]，再轉成[B,T,Da]，Da是音訊特徵寬度。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.multimodal import tone, log_mel, AudioEncoder

wave = tone()[None]
spectrogram = log_mel(wave)
encoder = AudioEncoder(bands=16, width=8)
features = encoder(wave)
print("頻譜", tuple(spectrogram.shape))
print("時間特徵", tuple(features.shape))

輸入預設0.1秒單音，`[None]`增加一段錄音的批次軸，成為(1,1600)。頻譜輸出(1,16,11)，表示16區段、11框。編碼器內部先做同樣log-mel、交換最後兩軸，再用16進8出的線性層及特徵處理；最終(1,11,8)。框數11保留，16是輸入區段數，8是每框特徵數。

當前權重隨機，因此這些8維數字還沒有明確的任務意義。可以像[10.5的視覺分類](https://birdhackor.github.io/tiny-perceptron-vlm/10.5.html)一樣，在特徵上加分類頭，教高低音或時長，觀察梯度回到編碼器，再用保留頻率與時長組合測量。分類頭可以之後拿掉，留下特徵接語言模型；形狀正確只是第一步。

語言token是把文字切分後交給模型的一個單位，可能是一個字、一個詞或詞的一部分，見[6.1的文字單位](https://birdhackor.github.io/tiny-perceptron-vlm/6.1.html)。每框特徵也不與語言token一一對應。一個字的發音可能跨許多框，一個框也可能處在字間；本章單音任務更根本沒有詞句。先從可核對高低音任務開始，是為了分清聲學輸入、特徵與語言輸出，而不是暗示11個框對應11個字。

轉軸並不是轉寫語音。把第0框16個mel區段放成一排、第1框另放一排，是把同一份數值改成方便序列處理的排列，沒有新增識別出的字詞。共享線性層讓每框用相同配方，而框順序保留聲音先後。若任務問時長，可由框數或時序變化提供線索；若只把全部框平均成一條特徵，某些順序與持續時間資訊可能被壓縮。之後使用什麼匯整方式，應由問題決定，這和圖像問左右時不能隨便平均位置是相同原則。

練習只把width8改12，先預測頻譜仍(1,16,11)、特徵變(1,11,12)，再執行核對。再只把`AudioEncoder(bands=16, width=8)`的bands改成32，特徵仍能輸出(1,11,8)：編碼器會一起把內部log-mel與線性層入口改為32區段。獨立展示的`spectrogram = log_mel(wave)`沒有跟著改，仍印(1,16,11)；若要讓展示也對應編碼器，另改成`log_mel(wave, bands=32)`，就印(1,32,11)。這两次log-mel各自計算，不是把展示用的16區段表送進32入口，不能預告只改bands就會報錯。
